In [1]:
import torch


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/sanaurooj/Library/Python/3.12/lib/python/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/sanaurooj/Library/Python/3.12/lib/python/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/sanaurooj/Library/Python/3.12/lib/python/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_lo

In [2]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt

In [3]:
import torch.nn as nn 
import torch.optim as optim 

## XOR classifier problem

In [39]:
X = torch.tensor([[0,0],
                 [0,1],
                 [1,0],
                 [1,1]],dtype=torch.float32)

y = torch.tensor([[0],
                  [1],
                  [1],
                  [0]],dtype=torch.float32)

print(f"x: {X}")
print(f"y: {y}")
print(f"y shape: {y.shape}")

x: tensor([[0., 0.],
        [0., 1.],
        [1., 0.],
        [1., 1.]])
y: tensor([[0.],
        [1.],
        [1.],
        [0.]])
y shape: torch.Size([4, 1])


In [40]:
class XORModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 4)   # 2 features and 4 hidden layers
        self.fc2 = nn.Linear(4, 1)   # 4 hidden layers and 1 output layer

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))

        return x

In [41]:
model_0 = XORModel()
model_0(X)

tensor([[0.5740],
        [0.5165],
        [0.5782],
        [0.5161]], grad_fn=<SigmoidBackward0>)

In [42]:
model_0(X).shape

torch.Size([4, 1])

In [43]:
model_0.fc1.bias

Parameter containing:
tensor([-0.0740,  0.3078,  0.5568,  0.4290], requires_grad=True)

In [44]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model_0.parameters(), lr=0.05)

In [45]:
for epoch in range(3000):
    #1.forward pass
    prediction = model_0(X)

    #2. loss calculation
    loss = criterion(prediction, y)

    #3. zero gradients
    optimizer.zero_grad()

    #4. backward propagation
    loss.backward()

    #5. update weights
    optimizer.step()

    if(epoch+1)%500==0:
        print(f"Epoch: {epoch+1} | Loss: {loss:.10f}")

print("\n── Results ──")
model_0.eval()
with torch.no_grad():
    raw = model_0(X)
    predicted = (raw >= 0.5).float()   # threshold at 0.5

    print(f"X shape: {X.shape}")
    print(f"raw shape: {raw.shape}")
    for i in range(4):
        inp  = X[i].int().tolist()
        x1 = int(X[i][0].item())           
        x2 = int(X[i][1].item()) 
        prob = raw[i].item()
        pred = int(predicted[i].item())
        true = int(y[i].item())
        tick = "✓" if pred == true else "✗"
        print(f"  {x1} XOR {x2} → prob={prob:.3f}  pred={pred}  true={true}  {tick}")

    acc = (predicted == y).float().mean().item()
    print(f"\nAccuracy: {acc*100:.0f}%")

Epoch: 500 | Loss: 0.0008896752
Epoch: 1000 | Loss: 0.0002687218
Epoch: 1500 | Loss: 0.0001285564
Epoch: 2000 | Loss: 0.0000737918
Epoch: 2500 | Loss: 0.0000466704
Epoch: 3000 | Loss: 0.0000313059

── Results ──
X shape: torch.Size([4, 2])
raw shape: torch.Size([4, 1])
  0 XOR 0 → prob=0.000  pred=0  true=0  ✓
  0 XOR 1 → prob=1.000  pred=1  true=1  ✓
  1 XOR 0 → prob=1.000  pred=1  true=1  ✓
  1 XOR 1 → prob=0.000  pred=0  true=0  ✓

Accuracy: 100%


In [46]:
model_0.state_dict()

OrderedDict([('fc1.weight',
              tensor([[ 0.0237, -0.3320],
                      [ 3.0027, -3.0028],
                      [ 4.5679,  2.2793],
                      [-3.6007, -3.6009]])),
             ('fc1.bias',
              tensor([-7.3972e-02,  1.5717e-05, -2.2803e+00,  3.6010e+00])),
             ('fc2.weight', tensor([[-0.4621,  4.3350, -4.5659, -5.6488]])),
             ('fc2.bias', tensor([9.2863]))])

In [47]:
!pip3 install torchinfo

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: /Library/Frameworks/Python.framework/Versions/3.12/bin/python3.12 -m pip install --upgrade pip


In [12]:
from torchinfo import summary

summary(XORModel())

Layer (type:depth-idx)                   Param #
XORModel                                 --
├─Linear: 1-1                            12
├─Linear: 1-2                            5
Total params: 17
Trainable params: 17
Non-trainable params: 0

## Regression : Fit a Sine wave

In [13]:
X = torch.linspace(-3.14, 3.14, 1000).unsqueeze(1)
y = torch.sin(X) + 0.1 * torch.randn_like(X)

In [14]:
y.shape

torch.Size([1000, 1])

In [15]:
class SineModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(1, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [16]:
model = SineModel()

In [17]:
model(X)

tensor([[-0.3917],
        [-0.3910],
        [-0.3903],
        [-0.3897],
        [-0.3890],
        [-0.3884],
        [-0.3877],
        [-0.3870],
        [-0.3864],
        [-0.3857],
        [-0.3851],
        [-0.3844],
        [-0.3837],
        [-0.3831],
        [-0.3824],
        [-0.3817],
        [-0.3811],
        [-0.3804],
        [-0.3798],
        [-0.3791],
        [-0.3784],
        [-0.3778],
        [-0.3771],
        [-0.3765],
        [-0.3758],
        [-0.3751],
        [-0.3745],
        [-0.3738],
        [-0.3732],
        [-0.3725],
        [-0.3718],
        [-0.3710],
        [-0.3703],
        [-0.3696],
        [-0.3689],
        [-0.3682],
        [-0.3675],
        [-0.3668],
        [-0.3661],
        [-0.3654],
        [-0.3647],
        [-0.3639],
        [-0.3632],
        [-0.3625],
        [-0.3618],
        [-0.3611],
        [-0.3604],
        [-0.3597],
        [-0.3590],
        [-0.3583],
        [-0.3576],
        [-0.3568],
        [-0.

In [18]:
model.fc1.bias.shape

torch.Size([64])

In [19]:
model.parameters

<bound method Module.parameters of SineModel(
  (fc1): Linear(in_features=1, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=1, bias=True)
)>

In [20]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=200, min_lr=1e-5, factor=0.5, verbose=True)

In [21]:
for epoch in range(2000):
    #1. forward pass
    prediction = model(X)

    #2. loss calculation 
    loss = criterion(prediction, y)

    #3. gradient zero
    optimizer.zero_grad()
    
    #4. backward pass
    loss.backward()

    #5. upgrade weights
    optimizer.step()

    # #6. learning rate scheduling
    # scheduler.step(loss)

    if((epoch+1)%500==0):
        print(f"Epoch: {epoch+1} | Loss: {loss:.10f} ")


Epoch: 500 | Loss: 0.0102170268 
Epoch: 1000 | Loss: 0.0101170484 
Epoch: 1500 | Loss: 0.0100927195 
Epoch: 2000 | Loss: 0.0100768274 


In [22]:
summary(SineModel())

Layer (type:depth-idx)                   Param #
SineModel                                --
├─Linear: 1-1                            128
├─Linear: 1-2                            4,160
├─Linear: 1-3                            65
Total params: 4,353
Trainable params: 4,353
Non-trainable params: 0

## Saving the XOR Model

In [ ]:
from pathlib import Path

# creating the directory
MODEL_PATH = Path('models')
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# creating the model save path
MODEL_NAME = "XOR_model.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

# saving the model state dict to the path , we can also save whole model but it is not recommended
print(f"Saving the model state dict to: {MODEL_SAVE_PATH}")
torch.save(model_0.state_dict(), MODEL_SAVE_PATH)

Saving the model state dict to: models/XOR_model.pth


## Loading a model

In [49]:
# creating an instance of the XORModel()
loaded_model_0 = XORModel()

# Load the saved state dict
loaded_model_0.load_state_dict(torch.load(MODEL_SAVE_PATH))

<All keys matched successfully>

In [50]:
loaded_model_0.state_dict()

OrderedDict([('fc1.weight',
              tensor([[ 3.0083, -2.9493],
                      [ 5.4284, -5.4283],
                      [-0.6307, -0.2659],
                      [-0.6286,  0.4315]])),
             ('fc1.bias',
              tensor([ 2.9496e+00,  4.6593e-05, -1.2593e-01, -5.3272e-01])),
             ('fc2.weight', tensor([[-6.7045,  7.6611,  0.0657,  0.2138]])),
             ('fc2.bias', tensor([9.2752]))])

In [51]:
# make some prediction with loaded state dict
loaded_model_0.eval()
with torch.inference_mode():
    loaded_model_preds = loaded_model_0(torch.tensor([0.0,1.0]))

loaded_model_preds

tensor([0.9999])